### Baseline Model

Based on the EDA.ipynb we can safely start with a baseline of a yes/no model

- Input: Image and Question
- Output: Probability of yes or no

In [3]:
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import torch.nn as nn

from transformers import AutoTokenizer, AutoModel

from datasets import load_dataset

In [14]:
def print_device_info():
    print("PyTorch version:", torch.__version__)
    if torch.cuda.is_available():
        print("CUDA available:", torch.cuda.is_available())
        print("Number of GPUs:", torch.cuda.device_count())
        print("Current device index:", torch.cuda.current_device())
        print("Current device name:", torch.cuda.get_device_name(0))
        device = torch.device("cuda")
    else:
        print("CUDA not available, using CPU.")
        device = torch.device("cpu")
    return device

device = print_device_info()

PyTorch version: 2.11.0+cpu
CUDA not available, using CPU.


In [2]:
ds = load_dataset("flaviagiammarino/vqa-rad")
train = ds["train"]
test = ds["test"]

def yes_no(example):
    return example["answer"].lower() in ["yes", "no"]

train_yn = train.filter(yes_no)
test_yn = test.filter(yes_no)

# Check the length of the datasets
print(len(train_yn), len(test_yn))

Filter: 100%|██████████| 451/451 [00:00<00:00, 558.34 examples/s]

940 251


The number of Yes and No in the train and test are accurate

In [4]:
label_map = {"yes": 1, "no": 0}
train_yn = train_yn.map(lambda ex: {"label": label_map[ex["answer"].lower()]})
test_yn = test_yn.map(lambda ex: {"label": label_map[ex["answer"].lower()]})

Map: 100%|██████████| 251/251 [00:00<00:00, 6545.94 examples/s]


Based on EDA
- Widths cluster roughly around 500–600 px and around 1000 px, with a few very large outliers
- Heights cluster around 400–800 px, with some up to around 1400 px

for the Baseline 224x224 is a standard size used by most CNNs such as ResNet

Downsizing from larger radiology images may:
- Loses some fine detail but for first experiments it is common to keep the compute cheap by keeping it to 224x224

We could be more careful by adding a CenterCrop

In [5]:
# Img_tranform with CenterCrop and Resize
img_transform = transforms.Compose([
    transforms.Resize(512),  # resize shortest side to 512 keeping the aspect ratio
    transforms.CenterCrop(512),  
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

In [6]:
# text
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
text_model = AutoModel.from_pretrained("distilbert-base-uncased")
text_model.eval() # Freezing this for baseline
for p in text_model.parameters():
    p.requires_grad = False

c:\Users\M333892\Desktop\VQA_Chest\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\M333892\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 11108.68it/s]
DistilBertModel LOAD REPORT from:

In [7]:
class VQARADBinaryImageText(Dataset):
    def __init__(self, hf_split, img_tranform, tokenizer, text_model, max_length=32):
        self.data = hf_split
        self.img_transform = img_tranform
        self.tokenizer = tokenizer
        self.text_model = text_model
        self.max_length = max_length # Based on EDA most of the questions are short so for Baseline 32 should be good 

    def __len__(self):
        return len(self.data)

    # Tokenize the question and get the CLS token embedding as the question representation
    def encode_question(self, question):
        inputs = self.tokenizer(question,
                                return_tensors="pt",
                                truncation=True,
                                padding="max_length",
                                max_length=self.max_length)
        with torch.no_grad():
            outputs = self.text_model(**inputs)
        cls_embedding = outputs.last_hidden_state[:, 0, :]  # (1, hidden_size)
        return cls_embedding.squeeze(0)  # (hidden_size,)

    def __getitem__(self, idx):
        ex = self.data[idx]
        image = ex["image"]
        if self.img_transform:
            img_tensor = self.img_transform(image)  # (3, 224, 224)
        
        question = ex["question"]
        text_embedding = self.encode_question(question)  # (hidden_dim,)

        label = ex["label"]  # 0 or 1
        return img_tensor, text_embedding, label

In [13]:
# To check if the dataset class is working correctly
dataset = VQARADBinaryImageText(train_yn, img_transform, tokenizer, text_model)
img, text_emb, label = dataset[3]
print("img shape:", img.shape)
print("text_emb shape:", text_emb.shape)
print("label:", label)

img shape: torch.Size([3, 224, 224])
text_emb shape: torch.Size([768])
label: 1


Utilizing the computing power of my PC at home

In [ ]:
train_dataset = VQARADBinaryImageText(train_yn, img_transform, tokenizer, text_model)
test_dataset = VQARADBinaryImageText(test_yn, img_transform, tokenizer, text_model)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

Build CNN and a MLP over concatenated features